In [1]:
import numpy as np 
import pandas as pd
mens_teams = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MTeams.csv")
seeds_men = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeeds.csv")
mens_tourney_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv")
mens_season_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonCompactResults.csv")


womens_teams = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WTeams.csv")
seeds_women = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySeeds.csv")
womens_tourney_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyCompactResults.csv")
womens_season_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv")

massey_ordinals =pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/MMasseyOrdinals.csv") #tbh idk what massey even is.. i dont speak basketballenese
stage_1_submission = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage1.csv")
stage_2_submission = pd.read_csv("/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv")


In [2]:
stage_1_submission.head()

,ID,Pred
0,2022_1101_1102,0.5
1,2022_1101_1103,0.5
2,2022_1101_1104,0.5
3,2022_1101_1105,0.5
4,2022_1101_1106,0.5


In [3]:
from pathlib import Path


STAGE = 1 #choose the stage submission as 2 for final submission :D
OUT_DIR = Path("/kaggle/working")

def parse_seed(s):
    if pd.isna(s):
        return np.nan
    return int("".join([c for c in str(s) if c.isdigit()]))

if "SeedNum" not in seeds_men.columns:
    seeds_men["SeedNum"] = seeds_men["Seed"].apply(parse_seed)
if "SeedNum" not in seeds_women.columns:
    seeds_women["SeedNum"] = seeds_women["Seed"].apply(parse_seed)

def build_season_stats(reg_df):
    w = reg_df[["Season","WTeamID","WScore","LScore"]].copy()
    w.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    w["Win"] = 1
    l = reg_df[["Season","LTeamID","LScore","WScore"]].copy()
    l.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    l["Win"] = 0
    df = pd.concat([w, l], ignore_index=True)
    df["Margin"] = df["ScoreFor"] - df["ScoreAgainst"]
    stats = df.groupby(["Season","TeamID"]).agg(
        Games=("Win","count"),
        Wins=("Win","sum"),
        AvgScore=("ScoreFor","mean"),
        AvgMargin=("Margin","mean")
    ).reset_index()
    stats["WinRate"] = stats["Wins"] / stats["Games"]
    return stats

m_stats = build_season_stats(mens_season_results)
w_stats = build_season_stats(womens_season_results)

massey_mor = (
    massey_ordinals[massey_ordinals["SystemName"] == "MOR"]
    .sort_values(["Season","RankingDayNum"])
    .groupby(["Season","TeamID"]).last()
    .reset_index()[["Season","TeamID","OrdinalRank"]]
    .rename(columns={"OrdinalRank":"MOR"})
)

def build_team_features(stats, seeds, mor_df=None):
    df = stats.merge(seeds[["Season","TeamID","SeedNum"]], on=["Season","TeamID"], how="left")
    if mor_df is not None:
        df = df.merge(mor_df, on=["Season","TeamID"], how="left")
    else:
        df["MOR"] = np.nan
    return df

m_feats = build_team_features(m_stats, seeds_men, massey_mor)
w_feats = build_team_features(w_stats, seeds_women, None)

def build_matchup_df(df_ids, team_feats, is_men, has_label):
    df = df_ids.copy()
    if has_label:
        df["Label"] = df["Label"].astype(int)
    df["is_men"] = is_men

    df = df.merge(team_feats, left_on=["Season","T1"], right_on=["Season","TeamID"], how="left")
    df = df.merge(team_feats, left_on=["Season","T2"], right_on=["Season","TeamID"], how="left", suffixes=("_T1","_T2"))
    df = df.drop(columns=["TeamID_T1","TeamID_T2"])

    for col in ["WinRate","AvgMargin","AvgScore","SeedNum","MOR","Games","Wins"]:
        if f"{col}_T1" in df.columns and f"{col}_T2" in df.columns:
            df[f"{col}Diff"] = df[f"{col}_T1"] - df[f"{col}_T2"]

    return df

def build_train_from_tourney(tourney, team_feats, is_men):
    t1 = np.minimum(tourney["WTeamID"], tourney["LTeamID"])
    t2 = np.maximum(tourney["WTeamID"], tourney["LTeamID"])
    label = (tourney["WTeamID"] == t1).astype(int)
    df_ids = pd.DataFrame({"Season": tourney["Season"], "T1": t1, "T2": t2, "Label": label})
    return build_matchup_df(df_ids, team_feats, is_men, has_label=True)

train_m = build_train_from_tourney(mens_tourney_results, m_feats, 1)
train_w = build_train_from_tourney(womens_tourney_results, w_feats, 0)
train_df = pd.concat([train_m, train_w], ignore_index=True)

sub = (stage_1_submission if STAGE == 1 else stage_2_submission).copy()
sub[["Season","T1","T2"]] = sub["ID"].str.split("_", expand=True).astype(int)

m_team_ids = set(mens_teams["TeamID"])
sub["Gender"] = sub["T1"].apply(lambda x: "M" if x in m_team_ids else "W")

test_m = build_matchup_df(sub[sub["Gender"]=="M"][["ID","Season","T1","T2"]], m_feats, 1, has_label=False)
test_w = build_matchup_df(sub[sub["Gender"]=="W"][["ID","Season","T1","T2"]], w_feats, 0, has_label=False)
test_df = pd.concat([test_m, test_w], ignore_index=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(OUT_DIR / "train.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print(f"Wrote {OUT_DIR / 'train.csv'} ({len(train_df):,} rows)")
print(f"Wrote {OUT_DIR / 'test.csv'} ({len(test_df):,} rows)")


Wrote /kaggle/working/train.csv (4,302 rows)
Wrote /kaggle/working/test.csv (519,144 rows)


In [4]:
train_df.tail(10)

,Season,T1,T2,Label,is_men,Games_T1,Wins_T1,AvgScore_T1,AvgMargin_T1,WinRate_T1,...,WinRate_T2,SeedNum_T2,MOR_T2,WinRateDiff,AvgMarginDiff,AvgScoreDiff,SeedNumDiff,MORDiff,GamesDiff,WinsDiff
4292,2025,3323,3395,0,0,31,26,84.709677,22.580645,0.838710,...,0.911765,2.0,NaN,-0.073055,2.286528,7.121442,1.0,NaN,-3,-5
4293,2025,3397,3400,0,0,31,22,87.193548,16.774194,0.709677,...,0.911765,1.0,NaN,-0.202087,-6.255218,8.252372,4.0,NaN,-3,-9
4294,2025,3243,3425,0,0,33,26,79.272727,20.727273,0.787879,...,0.903226,1.0,NaN,-0.115347,-0.885630,-2.178886,4.0,NaN,2,-2
4295,2025,3181,3376,0,0,33,26,73.909091,15.727273,0.787879,...,0.909091,1.0,NaN,-0.121212,-7.000000,-6.575758,1.0,NaN,0,-4
4296,2025,3261,3417,0,0,33,28,84.454545,18.787879,0.848485,...,0.937500,1.0,NaN,-0.089015,-2.180871,5.829545,2.0,NaN,1,-2
4297,2025,3163,3425,1,0,34,31,80.823529,28.970588,0.911765,...,0.903226,1.0,NaN,0.008539,7.357685,-0.628083,1.0,NaN,3,3
4298,2025,3395,3400,0,0,34,31,77.588235,20.294118,0.911765,...,0.911765,1.0,NaN,0.000000,-2.735294,-1.352941,1.0,NaN,0,0
4299,2025,3163,3417,1,0,34,31,80.823529,28.970588,0.911765,...,0.937500,1.0,NaN,-0.025735,8.001838,2.198529,1.0,NaN,2,1
4300,2025,3376,3400,1,0,33,30,80.484848,22.727273,0.909091,...,0.911765,1.0,NaN,-0.002674,-0.302139,1.543672,0.0,NaN,-1,-1
4301,2025,3163,3376,1,0,34,31,80.823529,28.970588,0.911765,...,0.909091,1.0,NaN,0.002674,6.243316,0.338681,1.0,NaN,1,1


In [5]:
from sklearn.metrics import log_loss
import lightgbm as lgb

OUT_DIR = Path("/kaggle/working")
train_df = pd.read_csv(OUT_DIR / "train.csv")
test_df = pd.read_csv(OUT_DIR / "test.csv")

target_col = "Label"
drop_cols = {"Label","ID"}
feature_cols = [c for c in train_df.columns if c not in drop_cols]

train_mask = train_df["Season"] < 2025
val_mask = train_df["Season"] == 2025

X_train = train_df.loc[train_mask, feature_cols]
y_train = train_df.loc[train_mask, target_col].astype(int)

X_val = train_df.loc[val_mask, feature_cols]
y_val = train_df.loc[val_mask, target_col].astype(int)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.0009,
    "num_leaves": 55,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.9,
    "bagging_freq": 3,
    "max_depth":15,
    "min_data_in_leaf": 5,
    "verbosity": -1,
    "seed": 42,
}

def brier_metric(preds, dataset):
    y_true = dataset.get_label()
    return "brier", float(np.mean((preds - y_true) ** 2)), False

dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val)

model = lgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    valid_sets=[dval],
    valid_names=["val"],
    feval=brier_metric,
    callbacks=[
        lgb.log_evaluation(period=50),
        lgb.early_stopping(stopping_rounds=300, first_metric_only=True)
    ]
)

val_pred = model.predict(X_val, num_iteration=model.best_iteration)
val_ll = log_loss(y_val, val_pred)
val_brier = np.mean((val_pred - y_val) ** 2)
print(f"Validation (2025) LogLoss: {val_ll:.6f}")
print(f"Validation (2025) Brier:   {val_brier:.6f}")

X_full = train_df[feature_cols]
y_full = train_df[target_col].astype(int)

dfull = lgb.Dataset(X_full, label=y_full)
model_full = lgb.train(
    params,
    dfull,
    num_boost_round=int(model.best_iteration or 500)
)

test_pred = model_full.predict(test_df[feature_cols])

sub = pd.DataFrame({"ID": test_df["ID"].values, "Pred": test_pred})
sub["Pred"] = sub["Pred"].clip(0.0001, 0.9999)

sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}")


Training until validation scores don't improve for 300 rounds
[50]	val's binary_logloss: 0.676691	val's brier: 0.24178
[100]	val's binary_logloss: 0.662406	val's brier: 0.234657
[150]	val's binary_logloss: 0.648982	val's brier: 0.227987
[200]	val's binary_logloss: 0.63594	val's brier: 0.221538
[250]	val's binary_logloss: 0.623887	val's brier: 0.215615
[300]	val's binary_logloss: 0.612672	val's brier: 0.210147
[350]	val's binary_logloss: 0.602021	val's brier: 0.204996
[400]	val's binary_logloss: 0.592224	val's brier: 0.200301
[450]	val's binary_logloss: 0.583333	val's brier: 0.196096
[500]	val's binary_logloss: 0.574835	val's brier: 0.19211
[550]	val's binary_logloss: 0.566681	val's brier: 0.188328
[600]	val's binary_logloss: 0.559167	val's brier: 0.184888
[650]	val's binary_logloss: 0.551869	val's brier: 0.181579
[700]	val's binary_logloss: 0.54502	val's brier: 0.178516
[750]	val's binary_logloss: 0.538532	val's brier: 0.17565
[800]	val's binary_logloss: 0.532579	val's brier: 0.173064
